# Feature value distributions by class

This notebook plots violin distributions of annotated matrix values for features `r`, `g`, `b`, `s`, and `d`, separately for the train, validation, and test splits.

Feature meanings:
- `r`: red channel
- `g`: green channel
- `b`: blue channel
- `s`: slope
- `d`: elevation

The plots use preprocessed pixel values from the annotated 20x20 matrices. NaN, infinite values, and exact `-999` sentinel values are treated as missing and replaced with the training-set mean for the corresponding feature before plotting. Each violin also shows the class mean with one standard deviation error bars.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Source.beach_dataset import (
    FEATURE_SUFFIXES,
    compute_channel_stats,
    load_cell_array,
    make_class_mapping,
    read_class_name_mapping,
    read_split_csv,
)

In [ ]:
ANNOTATED_ROOT = PROJECT_ROOT / "Data" / "Raw" / "Annotated_data"
CLASS_MAPPING_PATH = PROJECT_ROOT / "Data" / "Raw" / "Classes.xlsx"
PREPROCESSED_DIR = PROJECT_ROOT / "Data" / "Preprocessed"
SPLIT_FILES = {
    "train": PREPROCESSED_DIR / "train_split.csv",
    "validation": PREPROCESSED_DIR / "validation_split.csv",
    "test": PREPROCESSED_DIR / "test_split.csv",
}

RANDOM_SEED = 42
FEATURE_LABELS = {
    "r": "red channel (r)",
    "g": "green channel (g)",
    "b": "blue channel (b)",
    "s": "slope (s)",
    "d": "elevation (d)",
}
FEATURE_COLORS = {
    "r": "#d62728",
    "g": "#2ca02c",
    "b": "#1f77b4",
    "s": "#9467bd",
    "d": "#ff7f0e",
}

print(f"Annotated data: {ANNOTATED_ROOT}")
print(f"Features:       {FEATURE_SUFFIXES}")

In [ ]:
if not ANNOTATED_ROOT.exists():
    raise FileNotFoundError(f"Annotated data directory not found: {ANNOTATED_ROOT}")

splits = {split_name: read_split_csv(path) for split_name, path in SPLIT_FILES.items()}
class_to_idx = make_class_mapping(*splits.values())
class_name_by_code = read_class_name_mapping(CLASS_MAPPING_PATH, class_to_idx.keys())
class_codes = [code for code, _ in sorted(class_to_idx.items(), key=lambda item: item[1])]
class_labels = [f"{code}: {class_name_by_code.get(code, f'class_{code}')}" for code in class_codes]

channel_mean, channel_std = compute_channel_stats(splits["train"], ANNOTATED_ROOT, FEATURE_SUFFIXES)
channel_mean_by_feature = dict(zip(FEATURE_SUFFIXES, channel_mean))
channel_std_by_feature = dict(zip(FEATURE_SUFFIXES, channel_std))

summary = pd.DataFrame(
    [
        {"split": split_name, "rows": len(frame)}
        for split_name, frame in splits.items()
    ]
)
display(summary)
display(pd.DataFrame({"encoded_class": class_codes, "class_label": class_labels}))
display(
    pd.DataFrame(
        {
            "feature": FEATURE_SUFFIXES,
            "training_mean_used_for_missing_values": channel_mean,
            "training_std": channel_std,
        }
    )
)

In [ ]:
def collect_feature_samples(split_name, frame):
    shuffled = frame.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    buckets = {
        feature: {class_code: [] for class_code in class_codes}
        for feature in FEATURE_SUFFIXES
    }

    for row in tqdm(shuffled.itertuples(index=False), total=len(shuffled), desc=f"Loading {split_name}"):
        array = load_cell_array(ANNOTATED_ROOT, row.date, row.cell_id, FEATURE_SUFFIXES)
        class_code = str(row[2])

        for feature_index, feature in enumerate(FEATURE_SUFFIXES):
            values = array[feature_index].reshape(-1)
            feature_mean = channel_mean_by_feature[feature]
            values = np.where(np.isfinite(values), values, feature_mean)

            buckets[feature][class_code].extend(values.astype(float).tolist())

    return buckets


def plot_split_violins(split_name, buckets):
    fig, axes = plt.subplots(
        nrows=len(FEATURE_SUFFIXES),
        ncols=1,
        figsize=(max(12, len(class_codes) * 1.2), 3.2 * len(FEATURE_SUFFIXES)),
        sharex=True,
    )
    if len(FEATURE_SUFFIXES) == 1:
        axes = [axes]

    positions = np.arange(1, len(class_codes) + 1)
    for axis, feature in zip(axes, FEATURE_SUFFIXES):
        data = [buckets[feature][class_code] for class_code in class_codes]
        nonempty_positions = [pos for pos, values in zip(positions, data) if values]
        nonempty_data = [values for values in data if values]

        if nonempty_data:
            parts = axis.violinplot(
                nonempty_data,
                positions=nonempty_positions,
                showmeans=False,
                showmedians=True,
                showextrema=False,
            )
            for body in parts["bodies"]:
                body.set_facecolor(FEATURE_COLORS.get(feature, "#808080"))
                body.set_edgecolor("black")
                body.set_alpha(0.65)
            mean_values = np.array([np.mean(values) for values in nonempty_data])
            std_values = np.array([np.std(values) for values in nonempty_data])
            axis.errorbar(
                nonempty_positions,
                mean_values,
                yerr=std_values,
                fmt="o",
                color="black",
                ecolor="black",
                elinewidth=1.1,
                capsize=3,
                markersize=4,
                label="mean +/- 1 std",
            )
            if "cmedians" in parts:
                parts["cmedians"].set_color("black")
                parts["cmedians"].set_linewidth(1.2)

        feature_label = FEATURE_LABELS.get(feature, feature)
        axis.set_ylabel(feature_label)
        axis.grid(axis="y", alpha=0.25)
        axis.set_title(f"{split_name}: {feature_label}")

    axes[-1].set_xticks(positions)
    axes[-1].set_xticklabels(class_labels, rotation=45, ha="right")
    axes[-1].set_xlabel("Class")
    fig.suptitle(
        f"{split_name.capitalize()} feature-value distributions by class\n"
        "Missing values replaced with training feature means; bars show mean +/- 1 std",
        y=1.01,
    )
    fig.tight_layout()
    return fig

In [ ]:
train_buckets = collect_feature_samples("train", splits["train"])
train_fig = plot_split_violins("train", train_buckets)
plt.show()

In [ ]:
validation_buckets = collect_feature_samples("validation", splits["validation"])
validation_fig = plot_split_violins("validation", validation_buckets)
plt.show()

In [ ]:
test_buckets = collect_feature_samples("test", splits["test"])
test_fig = plot_split_violins("test", test_buckets)
plt.show()

In [ ]:
all_labeled_data = pd.concat(splits.values(), ignore_index=True)
all_buckets = collect_feature_samples("all labeled data", all_labeled_data)
all_fig = plot_split_violins("all labeled data", all_buckets)
plt.show()

In [ ]:
OUTLIER_THRESHOLD = -900

# Exact -999 sentinel values are converted to NaN by load_cell_array.
# This report catches any remaining finite slope/elevation values near -1000.

all_labeled_data = pd.concat(splits.values(), ignore_index=True)

outlier_rows = []
for _, row in tqdm(
    all_labeled_data.iterrows(),
    total=len(all_labeled_data),
    desc="Checking slope/elevation outliers",
):
    array = load_cell_array(ANNOTATED_ROOT, row["date"], row["cell_id"], FEATURE_SUFFIXES)
    slope = array[FEATURE_SUFFIXES.index("s")]
    elevation = array[FEATURE_SUFFIXES.index("d")]

    finite_slope = slope[np.isfinite(slope)]
    finite_elevation = elevation[np.isfinite(elevation)]
    slope_min = float(finite_slope.min()) if len(finite_slope) else np.nan
    elevation_min = float(finite_elevation.min()) if len(finite_elevation) else np.nan
    slope_outlier_pixels = int((np.isfinite(slope) & (slope <= OUTLIER_THRESHOLD)).sum())
    elevation_outlier_pixels = int((np.isfinite(elevation) & (elevation <= OUTLIER_THRESHOLD)).sum())
    has_slope_outlier = slope_outlier_pixels > 0
    has_elevation_outlier = elevation_outlier_pixels > 0

    if has_slope_outlier or has_elevation_outlier:
        class_code = str(row["class"])
        outlier_rows.append(
            {
                "split": row["split"],
                "date": row["date"],
                "cell_id": row["cell_id"],
                "class": class_code,
                "class_name": class_name_by_code.get(class_code, f"class_{class_code}"),
                "slope_min": slope_min,
                "elevation_min": elevation_min,
                "slope_outlier": has_slope_outlier,
                "elevation_outlier": has_elevation_outlier,
                "slope_outlier_pixels": slope_outlier_pixels,
                "elevation_outlier_pixels": elevation_outlier_pixels,
                "total_outlier_pixels": slope_outlier_pixels + elevation_outlier_pixels,
            }
        )

outlier_cells = pd.DataFrame(outlier_rows)
print(f"Outlier threshold: <= {OUTLIER_THRESHOLD}")
print(f"Cells with slope/elevation outliers: {len(outlier_cells):,}")
if outlier_cells.empty:
    total_slope_outlier_pixels = 0
    total_elevation_outlier_pixels = 0
else:
    total_slope_outlier_pixels = int(outlier_cells["slope_outlier_pixels"].sum())
    total_elevation_outlier_pixels = int(outlier_cells["elevation_outlier_pixels"].sum())
print(f"Slope outlier pixels: {total_slope_outlier_pixels:,}")
print(f"Elevation outlier pixels: {total_elevation_outlier_pixels:,}")
print(f"Total slope/elevation outlier pixels: {total_slope_outlier_pixels + total_elevation_outlier_pixels:,}")

if outlier_cells.empty:
    print("No slope/elevation outliers found.")
else:
    outlier_cells = outlier_cells.sort_values(["date", "cell_id"]).reset_index(drop=True)
    display(outlier_cells)
    print("Cell IDs:")
    print(", ".join(outlier_cells["cell_id"].astype(str).tolist()))

In [ ]:
if "outlier_cells" not in globals():
    raise RuntimeError("Run the slope/elevation outlier cell before this cell.")

outlier_keys = set()
if not outlier_cells.empty:
    outlier_keys = set(
        zip(
            outlier_cells["date"].astype(str),
            outlier_cells["cell_id"].astype(str),
        )
    )

all_labeled_data_without_outliers = all_labeled_data.loc[
    ~all_labeled_data.apply(
        lambda row: (str(row["date"]), str(row["cell_id"])) in outlier_keys,
        axis=1,
    )
].reset_index(drop=True)

print(f"Original labeled cells: {len(all_labeled_data):,}")
print(f"Excluded outlier cells: {len(outlier_keys):,}")
print(f"Remaining labeled cells: {len(all_labeled_data_without_outliers):,}")

all_without_outliers_buckets = collect_feature_samples(
    "all labeled data without slope/elevation outliers",
    all_labeled_data_without_outliers,
)
all_without_outliers_fig = plot_split_violins(
    "all labeled data without slope/elevation outliers",
    all_without_outliers_buckets,
)
plt.show()